# Single Image Processing from a LIF Container

This notebook processes 3D plant root images from LIF containers with the following steps:

1. Predicts 3D nuclei labels using CellposeSAM, including anisotropy correction and volume filtering.

All processing steps are visualized in Napari and results are saved to appropriate output folders.

Image Outputs are written to `RAW_DATA_DIRECTORY/<lif_container_id>/`.

### Environment setup
This cell sets up imports and makes sure the src folder is available in Python path.

Input needed from researcher: run this as-is unless your project folder layout is different.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import predict_nuclei_labels

from utils.feature_extraction import extract_nuclei_features_per_marker

from utils.data_viz import (
    map_df_column_to_labels,
    plot_prop_to_3d_centroids
)

import tifffile
import napari

### Configure experiment inputs
Set core paths and model choices used in the workflow.

Input needed from researcher: update RAW_DATA_DIRECTORY, MODEL_DIR, MIN_MAX_NUCLEI_VOLUME, channel indices, marker definitions, and index for image and container from your experiment.

In [ ]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"Z:\margrethe.c.stahl\Ki-67-proliferation"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 0

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels.
# Use a (min, max) voxel-count tuple, or None to skip volume filtering.
MIN_MAX_NUCLEI_VOLUME = None

# Marker information
MARKERS = (("DAPI", 0), ("F-Actin", 1), ("ki67", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 1 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 0

### Discover LIF containers
This cell scans the raw data directory and lists available .lif containers.

Input needed from researcher: confirm the directory contains the expected files.

In [ ]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

if not lif_containers:
    raise FileNotFoundError(
        f"No .lif containers found in '{RAW_DATA_DIRECTORY}'. "
        "Check RAW_DATA_DIRECTORY and file extension."
    )

if not 0 <= LIF_CONTAINER_INDEX < len(lif_containers):
    raise IndexError(
        f"LIF_CONTAINER_INDEX={LIF_CONTAINER_INDEX} is out of range. "
        f"Valid range is 0..{len(lif_containers)-1}."
    )

lif_containers

### Select one container and image
Choose the container and image index to analyze, then load metadata and image data.

Input needed from researcher: set LIF_CONTAINER_INDEX and LIF_IMAGE_INDEX to the sample you want to inspect in the cells above.

In [ ]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)


### Display input channels in Napari
This opens a 3D Napari view of all channels to verify image quality and channel order.

Input needed from researcher: visually confirm that channels and marker names match expectations.

In [ ]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'grey'],
                name=[tuple[0] for tuple in MARKERS]
                )

### Nuclei label prediction using CellposeSAM
This step loads existing nuclei labels if present, or predicts and saves new labels.

Input needed from researcher: verify CellposeSAM settings (for example anisotropy/rescale and channel choice).

In [ ]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



In [ ]:
### Alternative nuclei segmentation: Voronoi-Otsu (pyclesperanto)
GPU classical alternative to CellposeSAM. Resamples the nuclei channel to isotropic voxels (required by Voronoi-Otsu), labels with `cle.voronoi_otsu_labeling`, optionally size-filters, then maps labels back to the original grid for Napari overlay.

Input needed from researcher: tune `SPOT_SIGMA` (object separation) and `OUTLINE_SIGMA` (boundary smoothness); larger spot sigma merges nearby nuclei.

In [ ]:
import numpy as np
import pyclesperanto_prototype as cle
from skimage.transform import resize

# Tunable Voronoi-Otsu parameters
SPOT_SIGMA = 5      # larger -> fewer / more separated detected spots
OUTLINE_SIGMA = 1   # larger -> smoother object outlines

print("OpenCL devices:", cle.available_device_names())
print("Using device:", cle.select_device())

nuclei_channel = lif_image[NUCLEI_CHANNEL].astype(np.float32)

# Voronoi-Otsu assumes isotropic voxels: stretch Z by the anisotropy rescale factor
nuclei_iso = cle.scale(
    nuclei_channel,
    factor_x=1,
    factor_y=1,
    factor_z=float(rescale_factor),
    auto_size=True,
    linear_interpolation=True,
)

labels_iso = cle.voronoi_otsu_labeling(
    nuclei_iso,
    spot_sigma=SPOT_SIGMA,
    outline_sigma=OUTLINE_SIGMA,
)

# Optional volume filter (voxel counts are in the isotropic resampled space)
if MIN_MAX_NUCLEI_VOLUME is not None:
    min_size, max_size = MIN_MAX_NUCLEI_VOLUME
    labels_iso = cle.exclude_labels_out_of_size_range(
        labels_iso,
        minimum_size=min_size,
        maximum_size=max_size,
    )

# Map labels back to the original anisotropic grid (nearest-neighbor)
labels_iso_np = np.asarray(cle.pull(labels_iso))
nuclei_labels_voronoi = resize(
    labels_iso_np,
    nuclei_channel.shape,
    order=0,
    preserve_range=True,
    anti_aliasing=False,
).astype(np.uint32)

n_labels = int(nuclei_labels_voronoi.max())
print(f"Voronoi-Otsu nuclei counted: {n_labels}")

viewer.add_labels(nuclei_labels_voronoi, name="nuclei_voronoi_otsu")


### Extract nuclei features
Build image descriptors and compute per-nucleus morphological and intensity features.

Input needed from researcher: confirm marker/channel mapping reflects your biological labels.

In [ ]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            }

# Extract morphological and intensity features per marker
props_df = extract_nuclei_features_per_marker(nuclei_labels, lif_image, MARKERS, descriptor_dict)

### Visualize volumes
Render normalized volume per nucleus for exploratory interpretation.

Input needed from researcher: verify normalization and inspect outliers before analysis.

In [ ]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="area",
    colormap="inferno",
    visualize=True
)

### Matplotlib visualization

Besides Napari you can use Matplotlib for visualizing your results in a static graph

In [ ]:
plot_prop_to_3d_centroids(
    props_df,
    value_column="area",
    colormap="viridis",
    nuclei_labels_shape=nuclei_labels.shape,
    fig_title=None,
    ax_labels=None,
    colormap_vmin=None,
    colormap_vmax=None,
    save_fig=False,
    fig_filename=None,
    fig_savepath=None,
    visualize=True
)